# Simple Disaster COG Processing

> **Note:** Notebooks in this directory use CLI commands (`!command`) rather than Python library imports. This keeps workflows simple, reproducible, and avoids environment issues with GDAL/rasterio imports.

## Available CLI Tools
- `aws s3 ls / cp` - List and transfer S3 files
- `rio cogeo create` - Convert GeoTIFF to Cloud Optimized GeoTIFF
- `rio cogeo validate` - Validate COG structure
- `rio warp` - Reproject rasters
- `gdalinfo` - Inspect GeoTIFF metadata
- `process_landsat89` - Full Landsat 8/9 product generation pipeline
- `process_sentinel2` - Full Sentinel-2 product generation pipeline
- `download_sentinel2` - Download Sentinel-2 data from Copernicus

---

## Step 1: Configuration

Set your event details and S3 paths:

In [ ]:
# ========================================
# INPUTS - Edit these for your event
# ========================================

# S3 Paths
BUCKET = 'nasa-disasters'
DESTINATION_BASE = 'drcs_activations_new'
GEOTIFF_DIR = 'drcs_activations'

# Event Details
EVENT_NAME = '202510_Flood_AK'
SUB_PRODUCT_NAME = 'aria'
SOURCE_PATH = f'{GEOTIFF_DIR}/{EVENT_NAME}/{SUB_PRODUCT_NAME}'

# Processing Options
OVERWRITE = False
COMPRESSION = 'ZSTD'
COMPRESSION_LEVEL = 22
DST_CRS = 'EPSG:4326'

print(f"Event: {EVENT_NAME}")
print(f"Source: s3://{BUCKET}/{SOURCE_PATH}")
print(f"Destination: s3://{BUCKET}/{DESTINATION_BASE}/")

## Step 2: List S3 Files

See what files are available before processing:

In [ ]:
import subprocess, os

# List .tif files in the source path
result = subprocess.run(
    ['aws', 's3', 'ls', f's3://{BUCKET}/{SOURCE_PATH}/', '--recursive'],
    capture_output=True, text=True
)

files = []
if result.returncode == 0:
    for line in result.stdout.strip().split('\n'):
        if line.strip() and line.strip().endswith('.tif'):
            parts = line.strip().split()
            key = parts[-1]
            size_bytes = int(parts[2])
            size_gb = size_bytes / (1024**3)
            filename = os.path.basename(key)
            files.append(key)
            print(f"  {filename:<60} ({size_gb:.2f} GB)")
    print(f"\nFound {len(files)} .tif files")
else:
    print(f"Error listing files: {result.stderr}")

## Step 3: Define Filename Transformations

Based on the files listed above, configure how filenames should be renamed and where each category goes:

In [ ]:
import re

def extract_date_from_filename(filename):
    """Extract date from filename in YYYYMMDD format."""
    dates = re.findall(r'\d{8}', filename)
    if dates:
        d = dates[0]
        return f"{d[0:4]}-{d[4:6]}-{d[6:8]}"
    return None

def create_output_filename(original_path, event_name):
    """Create standardized output filename."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)
    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

# Categorization: regex pattern -> output subdirectory
CATEGORIES = {
    r'trueColor|truecolor|true_color': 'Landsat/trueColor',
    r'colorInfrared|colorIR|color_infrared': 'Landsat/colorIR',
    r'naturalColor|natural_color': 'Landsat/naturalColor',
    # Add more patterns as needed for your event
}

# Preview with a sample file
if files:
    sample = os.path.basename(files[0])
    new_name = create_output_filename(files[0], EVENT_NAME)
    matched = next((d for p, d in CATEGORIES.items() if re.search(p, sample, re.IGNORECASE)), 'uncategorized')
    print(f"Example: {sample}")
    print(f"     ->  {DESTINATION_BASE}/{matched}/{new_name}")

## Step 4: Preview All Transformations

In [ ]:
# Preview all file transformations before processing
processing_plan = []

for s3_key in files:
    filename = os.path.basename(s3_key)
    new_name = create_output_filename(s3_key, EVENT_NAME)

    # Match category
    output_dir = 'uncategorized'
    for pattern, directory in CATEGORIES.items():
        if re.search(pattern, filename, re.IGNORECASE):
            output_dir = directory
            break

    dest_key = f"{DESTINATION_BASE}/{output_dir}/{new_name}"
    processing_plan.append({'source': s3_key, 'dest': dest_key, 'filename': filename, 'new_name': new_name})
    print(f"  {filename}")
    print(f"    -> s3://{BUCKET}/{dest_key}\n")

print(f"Total: {len(processing_plan)} files to process")

## Step 5: Process Files (Download, COG Convert, Upload)

Downloads each file from S3, converts to COG using CLI tools, and uploads back:

In [ ]:
import time

results = []

for item in processing_plan:
    src_key = item['source']
    dest_key = item['dest']
    filename = item['filename']
    new_name = item['new_name']

    print(f"Processing: {filename}")
    start = time.time()

    # Check if destination already exists
    if not OVERWRITE:
        check = subprocess.run(
            ['aws', 's3', 'ls', f's3://{BUCKET}/{dest_key}'],
            capture_output=True, text=True
        )
        if check.returncode == 0 and check.stdout.strip():
            print(f"  SKIPPED (exists)\n")
            results.append({'file': filename, 'status': 'skipped', 'time': 0})
            continue

    local_input = f'/tmp/{filename}'
    local_warped = f'/tmp/warped_{new_name}'
    local_cog = f'/tmp/{new_name}'

    try:
        # 1. Download from S3
        subprocess.run(
            ['aws', 's3', 'cp', f's3://{BUCKET}/{src_key}', local_input],
            capture_output=True, text=True, check=True
        )

        # 2. Reproject if needed (gdalwarp with all-CPU threading)
        info = subprocess.run(['gdalinfo', '-json', local_input], capture_output=True, text=True)
        needs_reproject = DST_CRS.upper() not in info.stdout.upper()

        if needs_reproject:
            # gdalwarp chosen over `rio warp` so we can use NUM_THREADS=ALL_CPUS.
            subprocess.run([
                'gdalwarp',
                '-t_srs', DST_CRS,
                '-r', 'cubic',
                '-multi',
                '-wo', 'NUM_THREADS=ALL_CPUS',
                '-co', 'NUM_THREADS=ALL_CPUS',
                '--config', 'GDAL_NUM_THREADS', 'ALL_CPUS',
                '-overwrite',
                local_input, local_warped,
            ], capture_output=True, text=True, check=True)
            cog_input = local_warped
        else:
            cog_input = local_input

        # 3. Convert to COG (all-CPU)
        cmd = [
            'rio', 'cogeo', 'create', cog_input, local_cog,
            '--cog-profile', COMPRESSION.lower(),
            '--overview-level', '5',
            '--overview-resampling', 'average',
            '--nodata', '0',
            '--co', f'ZSTD_LEVEL={COMPRESSION_LEVEL}',
            '--co', 'PREDICTOR=2',
            '--co', 'NUM_THREADS=ALL_CPUS',
        ]
        subprocess.run(cmd, capture_output=True, text=True, check=True)

        # 4. Validate COG
        val = subprocess.run(
            ['rio', 'cogeo', 'validate', local_cog],
            capture_output=True, text=True
        )
        is_valid = val.returncode == 0

        # 5. Upload to S3
        subprocess.run(
            ['aws', 's3', 'cp', local_cog, f's3://{BUCKET}/{dest_key}'],
            capture_output=True, text=True, check=True
        )

        elapsed = time.time() - start
        status = 'success' if is_valid else 'success (COG validation warning)'
        print(f"  {status} ({elapsed:.1f}s)\n")
        results.append({'file': filename, 'status': status, 'time': elapsed})

    except subprocess.CalledProcessError as e:
        elapsed = time.time() - start
        print(f"  FAILED: {e.stderr[:200] if e.stderr else str(e)}\n")
        results.append({'file': filename, 'status': 'failed', 'time': elapsed, 'error': str(e)})

    finally:
        # Cleanup temp files
        for f in [local_input, local_warped, local_cog]:
            if os.path.exists(f):
                os.remove(f)

## Step 6: Results Summary

In [ ]:
total = len(results)
success = sum(1 for r in results if 'success' in r['status'])
failed = sum(1 for r in results if r['status'] == 'failed')
skipped = sum(1 for r in results if r['status'] == 'skipped')

print(f"Total files:  {total}")
print(f"  Success:    {success}")
print(f"  Failed:     {failed}")
print(f"  Skipped:    {skipped}")

if success > 0:
    times = [r['time'] for r in results if 'success' in r['status']]
    print(f"\nAvg time: {sum(times)/len(times):.1f}s per file")

if failed > 0:
    print("\nFailed files:")
    for r in results:
        if r['status'] == 'failed':
            print(f"  - {r['file']}: {r.get('error', 'Unknown')}")

## Troubleshooting

- **"No files found"** - Check `SOURCE_PATH`, verify bucket permissions, ensure `.tif` extension
- **Files being skipped** - Set `OVERWRITE = True` in Step 1
- **Processing errors** - Check `aws configure` has valid credentials, verify disk space in `/tmp`
- **Wrong CRS** - Inspect with `!gdalinfo /tmp/yourfile.tif` and adjust `DST_CRS`
- **COG validation warnings** - Usually harmless; check with `!rio cogeo validate /tmp/yourfile.tif`